In [ ]:
%pip install Pillow

In [ ]:
%pip install ipywidgets

In [ ]:
from pathlib import Path
import pymupdf
from PIL import Image
from IPython.display import display


In [ ]:
# Path handles file locations safely.
# PDF_PATH is the location of your test document.
# exists() checks the PDF is in the expected folder before we try to open it.

PROJECT_ROOT = Path.cwd().parent
PDF_PATH = PROJECT_ROOT / "samples" / "test.pdf"

print(PDF_PATH)
print("Exists:", PDF_PATH.exists())

In [ ]:
# age_count tells us how many pages there are

document = pymupdf.open(PDF_PATH)

print("Pages:", document.page_count)
print("Metadata:")
for key, value in document.metadata.items():
    print(f"{key}: {value}")

In [ ]:
# This is the foundation for: searching,creating an LLM summary,text-to-speech,indexing research papers,citation extraction.

page_number = 0
page = document[page_number]

text = page.get_text("text")

print(text[:2000])

In [ ]:
# PyMuPDF converts a PDF page into a pixmap—a grid of pixels—which we convert into a normal image for display

page = document[0]

pixmap = page.get_pixmap(dpi=150)

image = Image.frombytes(
    "RGB",
    (pixmap.width, pixmap.height),
    pixmap.samples
)

display(image)

In [ ]:
page_number = 1

if page_number < document.page_count:
    page = document[page_number]
    pixmap = page.get_pixmap(dpi=150)

    image = Image.frombytes(
        "RGB",
        (pixmap.width, pixmap.height),
        pixmap.samples
    )

    display(image)
else:
    print("This PDF does not have page 2.")

In [ ]:
# reusable page renderer

def render_page(document, page_number, dpi=150):
    if not 0 <= page_number < document.page_count:
        raise ValueError(
            f"Page number must be between 0 and {document.page_count - 1}"
        )

    page = document[page_number]
    pixmap = page.get_pixmap(dpi=dpi)

    mode = "RGBA" if pixmap.alpha else "RGB"

    image = Image.frombytes(
        mode,
        (pixmap.width, pixmap.height),
        pixmap.samples
    )

    return image

In [ ]:
current_page = 0

page_image = render_page(document, current_page, dpi=150)

print("Showing page:", current_page + 1)
print("Image size:", page_image.size)

display(page_image)

In [ ]:
# Explore PDF page properties
# A PDF page has a coordinate area called a rectangle. The reader will use this later for:
#     fitting a page to the windo,zooming,detecting where the user clicked,drawing highlights and annotations.

page = document[current_page]

print("Reader page number:", current_page + 1)
print("PDF rectangle:", page.rect)
print("Width in PDF points:", page.rect.width)
print("Height in PDF points:", page.rect.height)
print("Rotation:", page.rotation)

In [ ]:
# This is the beginning of your app’s internal state. A real reader needs to remember which document is open, 
# current page, zoom, selected theme, bookmarks, notes, and eventually the conversation/summary history for its LLM panel.

reader_state = {
    "file_path": str(PDF_PATH),
    "page_count": document.page_count,
    "current_page": 0,
    "zoom_dpi": 150,
    "theme": "dark",
    "bookmarks": [],
}

reader_state

In [ ]:
reader_state["current_page"] = 4
page_image = render_page(
    document,
    reader_state["current_page"],
    reader_state["zoom_dpi"]
)

display(page_image)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

reader_state = {
    "current_page": 0,
    "zoom_dpi": 120,
}

previous_button = widgets.Button(
    description="◀ Previous",
    button_style=""
)

next_button = widgets.Button(
    description="Next ▶",
    button_style=""
)

zoom_out_button = widgets.Button(
    description="− Zoom",
    button_style=""
)

zoom_in_button = widgets.Button(
    description="+ Zoom",
    button_style=""
)

page_label = widgets.Label()
zoom_label = widgets.Label()

output_area = widgets.Output()

toolbar = widgets.HBox([
    previous_button,
    next_button,
    zoom_out_button,
    zoom_in_button,
    page_label,
    zoom_label,
])

In [ ]:
%pip install --upgrade ipywidgets jupyterlab_widgets widgetsnbextension

In [ ]:
import ipywidgets as widgets
from IPython.display import display

display(widgets.IntSlider(
    value=50,
    min=0,
    max=100,
    description="Test:"
))

In [ ]:
def refresh_reader():
    current_page = reader_state["current_page"]
    dpi = reader_state["zoom_dpi"]

    page_label.value = (
        f"Page {current_page + 1} of {document.page_count}"
    )

    zoom_label.value = f"Render DPI: {dpi}"

    with output_area:
        clear_output(wait=True)

        image = render_page(document, current_page, dpi)
        display(image)

In [ ]:
refresh_reader()

In [ ]:
def show_previous_page(button):
    if reader_state["current_page"] > 0:
        reader_state["current_page"] -= 1
        refresh_reader()


def show_next_page(button):
    last_page = document.page_count - 1

    if reader_state["current_page"] < last_page:
        reader_state["current_page"] += 1
        refresh_reader()


def zoom_out(button):
    minimum_dpi = 72

    if reader_state["zoom_dpi"] > minimum_dpi:
        reader_state["zoom_dpi"] -= 24
        refresh_reader()


def zoom_in(button):
    maximum_dpi = 240

    if reader_state["zoom_dpi"] < maximum_dpi:
        reader_state["zoom_dpi"] += 24
        refresh_reader()


previous_button.on_click(show_previous_page)
next_button.on_click(show_next_page)
zoom_out_button.on_click(zoom_out)
zoom_in_button.on_click(zoom_in)

In [ ]:
display(toolbar)
display(output_area)

In [ ]:
page_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=document.page_count,
    step=1,
    description="Go to page:"
)

go_button = widgets.Button(
    description="Go",
    layout=widgets.Layout(width="70px")
)

def go_to_page(button):
    reader_state["current_page"] = page_input.value - 1
    refresh_reader()

go_button.on_click(go_to_page)

display(widgets.HBox([page_input, go_button]))

In [ ]:
def refresh_reader():
    current_page = reader_state["current_page"]
    dpi = reader_state["zoom_dpi"]

    page_label.value = (
        f"Page {current_page + 1} / {document.page_count}"
    )

    zoom_label.value = f"Render: {dpi} DPI"

    page_input.value = current_page + 1

    previous_button.disabled = (current_page == 0)
    next_button.disabled = (current_page == document.page_count - 1)

    with output_area:
        clear_output(wait=True)

        page_image = render_page(document, current_page, dpi)
        display(page_image)

In [ ]:
refresh_reader()

In [ ]:
search_term = "what is hci"

for page_number, page in enumerate(document):
    matches = page.search_for(search_term)

    if matches:
        print(
            f"Found {len(matches)} match(es) on "
            f"reader page {page_number + 1}"
        )

In [ ]:
from PIL import ImageDraw

In [ ]:
def render_page(document, page_number, dpi=120, highlight_rectangles=None):
    page = document[page_number]
    pixmap = page.get_pixmap(dpi=dpi)

    image_mode = "RGBA" if pixmap.alpha else "RGB"

    image = Image.frombytes(
        image_mode,
        (pixmap.width, pixmap.height),
        pixmap.samples
    )

    if highlight_rectangles:
        draw = ImageDraw.Draw(image, "RGBA")

        scale = dpi / 72

        for rect in highlight_rectangles:
            x0 = rect.x0 * scale
            y0 = rect.y0 * scale
            x1 = rect.x1 * scale
            y1 = rect.y1 * scale

            draw.rectangle(
                [x0, y0, x1, y1],
                outline=(255, 0, 0, 255),
                width=3
            )

            draw.rectangle(
                [x0, y0, x1, y1],
                fill=(255, 235, 0, 70)
            )

    return image

In [ ]:
search_input = widgets.Text(
    placeholder="Type a word or phrase...",
    description="Search:"
)

search_button = widgets.Button(
    description="Search",
    button_style="info"
)

next_match_button = widgets.Button(
    description="Next result ▶"
)

previous_match_button = widgets.Button(
    description="◀ Previous result"
)

search_status = widgets.Label(
    value="Search is ready."
)

search_toolbar = widgets.HBox([
    search_input,
    search_button,
    previous_match_button,
    next_match_button,
    search_status
])

display(search_toolbar)

In [ ]:
reader_state["search_results"] = []
reader_state["search_index"] = 0
reader_state["search_term"] = ""


def search_document(button=None):
    search_term = search_input.value.strip()

    if not search_term:
        search_status.value = "Enter a word or phrase first."
        return

    results = []

    for page_number, page in enumerate(document):
        rectangles = page.search_for(search_term)

        if rectangles:
            results.append({
                "page_number": page_number,
                "rectangles": rectangles
            })

    reader_state["search_term"] = search_term
    reader_state["search_results"] = results
    reader_state["search_index"] = 0

    if not results:
        search_status.value = f'No results for "{search_term}".'
        refresh_reader()
        return

    show_current_search_result()


def show_current_search_result():
    results = reader_state["search_results"]
    result_index = reader_state["search_index"]

    if not results:
        return

    result = results[result_index]
    reader_state["current_page"] = result["page_number"]

    search_status.value = (
        f'Result {result_index + 1} / {len(results)} '
        f'on page {result["page_number"] + 1}'
    )

    refresh_reader()


def next_search_result(button):
    results = reader_state["search_results"]

    if not results:
        search_status.value = "Search for something first."
        return

    reader_state["search_index"] = (
        reader_state["search_index"] + 1
    ) % len(results)

    show_current_search_result()


def previous_search_result(button):
    results = reader_state["search_results"]

    if not results:
        search_status.value = "Search for something first."
        return

    reader_state["search_index"] = (
        reader_state["search_index"] - 1
    ) % len(results)

    show_current_search_result()


search_button.on_click(search_document)
next_match_button.on_click(next_search_result)
previous_match_button.on_click(previous_search_result)

In [ ]:
def refresh_reader():
    current_page = reader_state["current_page"]
    dpi = reader_state["zoom_dpi"]

    page_label.value = (
        f"Page {current_page + 1} / {document.page_count}"
    )

    zoom_label.value = f"Render: {dpi} DPI"

    page_input.value = current_page + 1

    previous_button.disabled = (current_page == 0)
    next_button.disabled = (current_page == document.page_count - 1)

    highlight_rectangles = None

    results = reader_state.get("search_results", [])
    result_index = reader_state.get("search_index", 0)

    if results and results[result_index]["page_number"] == current_page:
        highlight_rectangles = results[result_index]["rectangles"]

    with output_area:
        clear_output(wait=True)

        page_image = render_page(
            document,
            current_page,
            dpi,
            highlight_rectangles
        )

        display(page_image)

In [ ]:
refresh_reader()

In [ ]:
display(search_toolbar)

In [1]:
from pathlib import Path

import pymupdf
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display, clear_output


# ---------- Open PDF ----------
PROJECT_ROOT = Path.cwd().parent
PDF_PATH = PROJECT_ROOT / "samples" / "test.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find PDF: {PDF_PATH}\n"
        "Check the filename and samples folder."
    )

document = pymupdf.open(PDF_PATH)


# ---------- Reader state ----------
reader_state = {
    "current_page": 0,
    "zoom_dpi": 120,
    "search_results": [],
    "search_index": 0,
}


# ---------- Render PDF page ----------
def render_page(page_number, dpi, highlight_rectangles=None):
    page = document[page_number]
    pixmap = page.get_pixmap(dpi=dpi)

    mode = "RGBA" if pixmap.alpha else "RGB"

    image = Image.frombytes(
        mode,
        (pixmap.width, pixmap.height),
        pixmap.samples
    )

    if highlight_rectangles:
        draw = ImageDraw.Draw(image, "RGBA")
        scale = dpi / 72

        for rect in highlight_rectangles:
            draw.rectangle(
                [
                    rect.x0 * scale,
                    rect.y0 * scale,
                    rect.x1 * scale,
                    rect.y1 * scale
                ],
                fill=(255, 235, 0, 90),
                outline=(255, 0, 0, 255),
                width=3
            )

    return image


# ---------- Create interface ----------
previous_button = widgets.Button(description="◀ Previous")
next_button = widgets.Button(description="Next ▶")

zoom_out_button = widgets.Button(description="− Zoom")
zoom_in_button = widgets.Button(description="+ Zoom")

page_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=document.page_count,
    description="Page:"
)

go_button = widgets.Button(description="Go")

page_label = widgets.Label()
zoom_label = widgets.Label()

search_input = widgets.Text(
    placeholder="Type a word or phrase...",
    description="Search:"
)

search_button = widgets.Button(
    description="Search",
    button_style="info"
)

previous_result_button = widgets.Button(
    description="◀ Previous result"
)

next_result_button = widgets.Button(
    description="Next result ▶"
)

search_status = widgets.Label(value="Search is ready.")

page_output = widgets.Output()

navigation_bar = widgets.HBox([
    previous_button,
    next_button,
    zoom_out_button,
    zoom_in_button,
    page_input,
    go_button,
    page_label,
    zoom_label
])

search_bar = widgets.HBox([
    search_input,
    search_button,
    previous_result_button,
    next_result_button,
    search_status
])


# ---------- Refresh the displayed page ----------
def refresh_reader():
    current_page = reader_state["current_page"]
    dpi = reader_state["zoom_dpi"]

    page_input.value = current_page + 1
    page_label.value = f"Page {current_page + 1} / {document.page_count}"
    zoom_label.value = f"Render: {dpi} DPI"

    previous_button.disabled = (current_page == 0)
    next_button.disabled = (current_page == document.page_count - 1)

    highlights = None
    results = reader_state["search_results"]

    if results:
        current_result = results[reader_state["search_index"]]

        if current_result["page_number"] == current_page:
            highlights = current_result["rectangles"]

    with page_output:
        clear_output(wait=True)
        display(render_page(current_page, dpi, highlights))


# ---------- Page navigation ----------
def show_previous_page(button):
    if reader_state["current_page"] > 0:
        reader_state["current_page"] -= 1
        refresh_reader()


def show_next_page(button):
    if reader_state["current_page"] < document.page_count - 1:
        reader_state["current_page"] += 1
        refresh_reader()


def go_to_page(button):
    reader_state["current_page"] = page_input.value - 1
    refresh_reader()


def zoom_out(button):
    if reader_state["zoom_dpi"] > 72:
        reader_state["zoom_dpi"] -= 24
        refresh_reader()


def zoom_in(button):
    if reader_state["zoom_dpi"] < 240:
        reader_state["zoom_dpi"] += 24
        refresh_reader()


# ---------- Search ----------
def search_document(button=None):
    term = search_input.value.strip()

    if not term:
        search_status.value = "Enter text to search."
        return

    results = []

    for page_number, page in enumerate(document):
        rectangles = page.search_for(term)

        if rectangles:
            results.append({
                "page_number": page_number,
                "rectangles": rectangles
            })

    reader_state["search_results"] = results
    reader_state["search_index"] = 0

    if not results:
        search_status.value = f'No results for "{term}".'
        refresh_reader()
        return

    show_current_result()


def show_current_result():
    result = reader_state["search_results"][reader_state["search_index"]]

    reader_state["current_page"] = result["page_number"]

    search_status.value = (
        f'Result {reader_state["search_index"] + 1} / '
        f'{len(reader_state["search_results"])} '
        f'on page {result["page_number"] + 1}'
    )

    refresh_reader()


def show_next_result(button):
    if not reader_state["search_results"]:
        search_status.value = "Search for a word first."
        return

    reader_state["search_index"] = (
        reader_state["search_index"] + 1
    ) % len(reader_state["search_results"])

    show_current_result()


def show_previous_result(button):
    if not reader_state["search_results"]:
        search_status.value = "Search for a word first."
        return

    reader_state["search_index"] = (
        reader_state["search_index"] - 1
    ) % len(reader_state["search_results"])

    show_current_result()


# ---------- Connect button events ----------
previous_button.on_click(show_previous_page)
next_button.on_click(show_next_page)
go_button.on_click(go_to_page)

zoom_out_button.on_click(zoom_out)
zoom_in_button.on_click(zoom_in)

search_button.on_click(search_document)
next_result_button.on_click(show_next_result)
previous_result_button.on_click(show_previous_result)


# ---------- Display complete reader ----------
display(navigation_bar)
display(search_bar)
display(page_output)

refresh_reader()

Output()